# Chapter 11 — Exercises: Worked Solutions

Worked solutions for Chapter 11 (CO\u2082 Systems and CCS).

**Exercises:**
1. CO\u2082 phase behavior and dense phase region
2. CO\u2082 solubility in water with CPA
3. Effect of impurities on CO\u2082 phase envelope

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")
R = 8.314

SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

---
## Exercise 11.1 — CO\u2082 Density Across the Phase Boundary

**Problem:** Calculate CO\u2082 density at 10\u00b0C for pressures from 10 to
200 bar. Identify the vapor\u2013liquid transition and the dense phase region.

### Solution

In [3]:
pressures = np.arange(10, 205, 2)
rho_co2 = []

for P in pressures:
    try:
        f = SystemSrkEos(273.15 + 10.0, float(P))
        f.addComponent("CO2", 1.0)
        f.setMixingRule("classic")
        ops = ThermodynamicOperations(f)
        ops.TPflash()
        f.initProperties()
        rho = float(f.getDensity("kg/m3"))
        rho_co2.append(rho)
    except Exception:
        rho_co2.append(np.nan)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(pressures[:len(rho_co2)], rho_co2, color=BLUE, linewidth=1.5)
ax.axvline(x=73.8, color="red", linestyle="--", alpha=0.5, linewidth=0.8)
ax.annotate("$P_c$ = 73.8 bar", xy=(76, 200), fontsize=7, color="red")
ax.set_xlabel("Pressure (bar)"); ax.set_ylabel("Density (kg/m\u00b3)")
ax.set_title("Exercise 11.1: CO\u2082 density at 10\u00b0C")
ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch11_ex01_co2_density.png")

Saved: fig_ch11_ex01_co2_density.png


**Answer:** At 10\u00b0C (below $T_c$ = 31.1\u00b0C), CO\u2082 shows a sharp
density jump at the vapor pressure (~45 bar) from gas (~100 kg/m\u00b3)
to liquid (~900 kg/m\u00b3). Above $P_c$, CO\u2082 is in the dense phase with
liquid-like density but no phase boundary — this is the regime used
for pipeline transport in CCS.

---
## Exercise 11.2 — CO\u2082 Solubility in Water

**Problem:** Compare CO\u2082 solubility in water at 25\u00b0C and 50\u00b0C
for pressures up to 100 bar using CPA.

### Solution

In [4]:
pressures_sol = np.arange(5, 105, 5)

fig, ax = plt.subplots(figsize=(3.5, 2.8))

for T_C, color, ls in [(25, BLUE, "-"), (50, ORANGE, "--")]:
    x_co2 = []
    for P in pressures_sol:
        try:
            f = SystemSrkCPAstatoil(273.15 + T_C, float(P))
            f.addComponent("CO2", 0.98)
            f.addComponent("water", 0.02)
            f.setMixingRule(10)
            f.setMultiPhaseCheck(True)
            ops = ThermodynamicOperations(f)
            ops.TPflash()
            f.initProperties()
            nph = int(f.getNumberOfPhases())
            found = False
            for ph in range(nph):
                pt = str(f.getPhase(ph).getPhaseTypeName()).lower()
                if "aqueous" in pt or ("liquid" in pt and ph > 0):
                    x_co2.append(float(f.getPhase(ph).getComponent("CO2").getx()) * 100)
                    found = True
                    break
            if not found:
                x_co2.append(np.nan)
        except Exception:
            x_co2.append(np.nan)
    ax.plot(pressures_sol[:len(x_co2)], x_co2, color=color, linestyle=ls, linewidth=1.5,
            label=f"{T_C}\u00b0C")

ax.set_xlabel("Pressure (bar)"); ax.set_ylabel("$x_{CO_2}$ in water (mol%)")
ax.set_title("Exercise 11.2: CO\u2082 solubility in water")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch11_ex02_co2_solubility.png")

Saved: fig_ch11_ex02_co2_solubility.png


**Answer:** CO\u2082 solubility increases with pressure and decreases with
temperature. At 25\u00b0C, the solubility is higher than at 50\u00b0C at all
pressures. The solubility levels off above the vapor pressure because
CO\u2082 transitions to a denser phase. CPA with solvation captures this
behavior accurately.

---
## Exercise 11.3 — Effect of N\u2082 Impurity on CO\u2082 Cricondenbar

**Problem:** Show how 2% and 5% N\u2082 impurity changes the phase envelope
of CO\u2082, particularly the cricondenbar.

### Solution

In [5]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

configs = [
    ("Pure CO\u2082", {"CO2": 1.0}, BLUE, "-"),
    ("2% N\u2082", {"CO2": 0.98, "nitrogen": 0.02}, ORANGE, "--"),
    ("5% N\u2082", {"CO2": 0.95, "nitrogen": 0.05}, GREEN, ":")
]

for label, comp, color, ls in configs:
    try:
        f = SystemSrkEos(273.15, 50.0)
        for name, frac in comp.items():
            f.addComponent(name, frac)
        f.setMixingRule("classic")
        ops = ThermodynamicOperations(f)
        ops.calcPTphaseEnvelope(True, 1.0)

        Ts = [float(x) - 273.15 for x in ops.getOperation().get("dewT")]
        Ps = [float(x) for x in ops.getOperation().get("dewP")]
        Tb = [float(x) - 273.15 for x in ops.getOperation().get("bubT")]
        Pb = [float(x) for x in ops.getOperation().get("bubP")]

        ax.plot(Ts, Ps, color=color, linestyle=ls, linewidth=1.2, label=label + " dew")
        ax.plot(Tb, Pb, color=color, linestyle=ls, linewidth=1.0, alpha=0.7)
    except Exception as e:
        print(f"Phase envelope failed for {label}: {e}")

ax.set_xlabel("Temperature (\u00b0C)"); ax.set_ylabel("Pressure (bar)")
ax.set_title("Exercise 11.3: N\u2082 impurity effect")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch11_ex03_n2_impurity.png")

Saved: fig_ch11_ex03_n2_impurity.png


**Answer:** N\u2082 impurity raises the cricondenbar pressure and creates
a two-phase region that pure CO\u2082 does not have above its critical
point. Even 2% N\u2082 can increase the cricondenbar by ~15 bar, meaning
CO\u2082 pipelines must operate at higher pressures to avoid two-phase
flow. This has direct implications for CCS pipeline design.